# Lecture 08: InstructGPT and reward models

Exercises E01–E04 follow the slides. Natural logarithms are used.

Ungraded practice. Python 3.11+, standard library only; offline CPU, 0 GPU-hours. Predict each result before running the checked example. All numerical data are synthetic.


In [ ]:
import math
from statistics import mean, pstdev


## E01 · 6 minutes · Mask and reduction

Response A has target probabilities (1/2, 1/4), response B has (1/4). Exclude prompt and padding targets. Compute the token mean and mean of response means.

The following cell checks the worked answer.


In [ ]:
responses = [[0.5, 0.25], [0.25]]
nll = [[-math.log(p) for p in row] for row in responses]
token_mean = sum(map(sum, nll)) / sum(map(len, nll))
response_mean = mean(mean(row) for row in nll)
assert math.isclose(token_mean, 5/3 * math.log(2))
assert math.isclose(response_mean, 7/4 * math.log(2))
print({'token_mean': token_mean, 'response_mean': response_mean})


## E02 · 5 minutes · Pairwise reward model

For rewards log(3) and 0, calculate the preferred-response probability and loss. Add 100 to both rewards; does the probability change?

The following cell checks the worked answer.


In [ ]:
def preference_probability(winner, loser):
    gap = winner - loser
    return 1 / (1 + math.exp(-gap))
p = preference_probability(math.log(3), 0)
assert math.isclose(p, 0.75)
assert math.isclose(-math.log(p), math.log(4/3))
assert math.isclose(preference_probability(math.log(3)+100, 100), p)
print({'preferred_probability': p, 'loss': -math.log(p)})


## E03 · 6 minutes · Split by prompt family

Two paraphrases from family A are in train and test. Family B is only in test. Name the leak and propose a family-disjoint split. Is one test family enough for a useful general claim?

The following cell checks the worked answer.


In [ ]:
records = [('a1', 'A', 'train'), ('a2', 'A', 'test'), ('b1', 'B', 'test')]
def crossed_families(rows):
    splits = {}
    for _, family, split in rows:
        splits.setdefault(family, set()).add(split)
    return {family for family, values in splits.items() if len(values) > 1}
assert crossed_families(records) == {'A'}
repaired = [(rid, family, 'train' if family == 'A' else 'test') for rid, family, _ in records]
assert not crossed_families(repaired)
print(repaired)
print('One held-out family cannot support broad generalization; collect representative independent families.')


## E04 · 5 minutes · Old policy and reference

One sampled token has old probability .20, fixed-reference probability .10, and current probability .24. Compute the PPO ratio and sampled reference log ratio.

The following cell checks the worked answer.


In [ ]:
old, reference, current = 0.20, 0.10, 0.24
ratio = current / old
sampled_log_ratio = math.log(current / reference)
assert math.isclose(ratio, 1.2)
assert math.isclose(sampled_log_ratio, math.log(2.4))
print({'ppo_ratio': ratio, 'reference_log_ratio': sampled_log_ratio})
print('This single-token log ratio is not a full KL divergence.')
